---
title: "Resultados potenciales"
description: "Modelo de resultados potenciales y mecanismo de asignación"
categories: [Causal Inference]
order: 1
---

In [2]:
import numpy as np
import pandas as pd

# Resultados potenciales

## Notación de resultados potenciales

<br>

**Definición: Resultados potenciales $Y_i(1)$ y $Y_i(0)$**

Consideremos un estudio con $n$ unidades cuyo índice es $i = 1, 2,...,n$ en donde el tratamiento tiene dos niveles:
 * $1$ para el tratamiento,
 * $0$ para el control.

Con esto, cada unidad $i$ tiene dos versiones del resultado de interés:
$$Y_i(1), Y_i(0)$$

los cuales son llamados *resultados potenciales* bajo las intervenciones $1$ y $0$, respectivamente.

<br>

**Supuestos**

La definición anterior conlleva dos supuestos.

***1. No interferencia***: Los resultados potenciales de la unidad $i$ no dependen del tratamiento asignado a ninguna otra unidad $j$. En otras palabras, lo que le pase a $j$ no afecta lo que le hubiera pasado a $i$.

***2. Consistencia***: No existen versiones múltiples del tratamiento; el tratamiento está bien definido. Si la unidad $i$ recibe el tratamiento, entonces su resultado observado es $Y_i(1)$.

Los supuestos 1 y 2 combinados forman el *Stable Unit Treatment Value Assumption* o **SUTVA**. Bajo SUTVA, los resultados potenciales de cada unidad son fijos y no dependen de la asignación de tratamiento de otras unidades, lo que hace posible construir la *tabla de ciencia*:

| $i$ | $Y_i(1)$ | $Y_i(0)$ |
|-----|----------|----------|
| 1 | $Y_1(1)$ | $Y_1(0)$ |
| 2 | $Y_2(1)$ | $Y_2(0)$ |
| $\vdots$ | $\vdots$ | $\vdots$ |
| $n$ | $Y_n(1)$ | $Y_n(0)$ |

La tabla de ciencia es un objeto conceptual: en la práctica, cada unidad solo puede recibir un nivel del tratamiento, por lo que solo una de las dos columnas es observable para cada fila (unidad).

<br>

**Definición: efecto causal individual $\tau_i$**

El efecto causal para la unidad $i$ se define como la diferencia entre sus dos resultados potenciales:

$$\tau_i = Y_i(1) - Y_i(0)$$

Este es el contraste fundamental de la inferencia causal. El problema es que nunca podemos observar $\tau_i$ directamente, porque solo observamos uno de los dos resultados potenciales: el que corresponde al tratamiento que efectivamente recibió la unidad. El resultado que no observamos se llama *contrafactual*.

<br>

**Definición: Efecto Causal Promedio (ATE) $\tau$**

Dado que $\tau_i$ no es identificable a nivel individual, el objeto de interés más común es el promedio sobre todas las unidades:

$$\tau = n^{-1}\sum_{i=1}^{n}\{Y_i(1) - Y_i(0)\} = \bar{Y}(1) - \bar{Y}(0)$$

donde $\bar{Y}(1)$ y $\bar{Y}(0)$ son los promedios poblacionales de los resultados potenciales bajo tratamiento y control, respectivamente.

## Efectos causales por subgrupo y la paradoja de Yule-Simpson

El ATE puede descomponerse como un promedio ponderado de los efectos causales en subgrupos de la población. Si $x$ indica la pertenencia a un grupo, entonces:

$$\tau = \sum_x \Pr(X=x) \cdot \tau_x$$

donde $\tau_x = \mathbb{E}[Y_i(1) - Y_i(0) \mid X_i = x]$ es el efecto causal promedio dentro del grupo $x$.

Esto es relevante porque la *paradoja de Yule-Simpson* surge cuando se comparan diferencias de medias en los datos observados sin controlar por covariables. Puede ocurrir que la dirección del efecto se invierta al agregar o desagregar grupos. Sin embargo, **en términos de resultados potenciales no existe tal paradoja**: el ATE siempre es el promedio ponderado de los efectos por subgrupo, independientemente de cómo estén distribuidos los grupos.

El siguiente ejemplo ilustra este punto: aunque los grupos difieren en su nivel de $Y_i(0)$, el efecto causal dentro de cada grupo es similar, y el ATE global es simplemente el promedio ponderado por tamaño de grupo.

In [3]:
# Efectos causales, subgrupos y la no existencia de la paradoja de Yule-Simpson

# En este ejemplo, vamos a calcular los efectos causales de subgrupos
# En este caso, necesitamos generar dependencia por grupo y generar dependencia
# del efecto potencial de aplicar el tratamiento, dado el efecto potencial del control

x = np.random.binomial(1, 0.5, 100)
y_0 = np.where(x==1, np.random.normal(8,1,100), np.random.normal(4,1,100) ) # dependencia e
y_1 = y_0 + np.random.normal(loc = 2, scale = 0.5, size = 100)
tau = y_1 - y_0
n = np.array([100 - np.sum(x), np.sum(x)])

df = pd.DataFrame({"y_1":y_1, "y_0":y_0, "x":x, "tau_grupo":tau})

df_summary = df.groupby("x").mean()
df_summary["n"] = n/np.sum(n)

# tau_grupo presenta el efecto causal promedio del subgrupo.
print("Efectos causales por grupo:\n",df_summary['tau_grupo'])

# El efecto causal promedio global es el promedio ponderado de tau_grupo
tau_global = np.sum(df_summary['n']*df_summary['tau_grupo'])
print("Efecto causal global:", float(tau_global))

Efectos causales por grupo:
 x
0    2.026268
1    1.925103
Name: tau_grupo, dtype: float64
Efecto causal global: 1.9787206915824778


## Mecanismo de asignación de tratamiento

Sea $Z_i$ una variable indicadora de tratamiento para la unidad $i$, vectorizada como $\mathbf{Z} = (Z_1, Z_2, ..., Z_n)^T$.

El resultado observado de la unidad $i$ es:

$$\begin{align}
Y_i &= \begin{cases} Y_i(1), & \text{si } Z_i = 1 \\ Y_i(0), & \text{si } Z_i = 0 \end{cases}\\
&= Z_i Y_i(1) + (1 - Z_i)Y_i(0)\\
&= Y_i(0) + Z_i\tau_i.
\end{align}$$

La última expresión deja ver que los efectos causales pueden ser heterogéneos entre unidades: el mismo tratamiento puede tener efectos distintos dependiendo de quién lo recibe.

Cuando se aplica el tratamiento, uno de los dos resultados potenciales se vuelve no observable. Ese resultado no observado es el *contrafactual*.

<br>

**Definición: mecanismo de asignación de tratamiento**

El mecanismo de asignación es la distribución de probabilidad condicional de $\mathbf{Z}$ dado los resultados potenciales y las covariables:

$$\Pr(\mathbf{Z} = \mathbf{z} \mid \mathbf{Y}(1), \mathbf{Y}(0), \mathbf{X})$$

El mecanismo de asignación determina qué tan difícil es estimar los efectos causales a partir de los datos observados. Si el mecanismo depende de los resultados potenciales —es decir, si quienes reciben el tratamiento son distintos de quienes no lo reciben en formas que también afectan el resultado— entonces la simple diferencia de medias entre tratados y controles no identifica el ATE.

In [4]:
# Ejemplo: Efectos potenciales y mecanismo de asignación de tratamiento

# Queremos mostrar el rol crucial del mecanismo de asignación de tratamiento
# sobre los estimadores de los efectos causales promedio

# Generamos los resultados potenciales y los efectos causales
n = 50
y_0 = np.random.normal(size=n)
tau = -0.5 + y_0
y_1 = y_0 + tau

# Doctor perfecto: asigna el tratamiento cuando el efecto causal es no negativo
z = np.where(tau >= 0, 1, 0) # Mecanismo de asignación, asigna cuando tau es mayor a 1
y = y_0 + z*tau # resultado observado

print("Diferencia de medias en el caso del doctor perfecto:",
      float(np.mean(y[z==1]) - np.mean(y[z==0])))

# Doctor ingenuo: asigna el tratamiento aleatoriamente
z = np.random.binomial(1, 0.5, n)
y = y_0 + z*tau # resultado observado

print("Diferencia de medias en el caso del doctor ingenuo:",
      float(np.mean(y[z==1]) - np.mean(y[z==0])))

Diferencia de medias en el caso del doctor perfecto: 2.691626430163731
Diferencia de medias en el caso del doctor ingenuo: -0.7993871776399679


En el ejemplo, $\tau_i = -0.5 + Y_i(0)$, por lo que el efecto del tratamiento es positivo solo cuando $Y_i(0) > 0.5$. El doctor perfecto asigna el tratamiento precisamente a esas unidades, que de por sí tienen resultados potenciales altos bajo control. La diferencia de medias observada entre tratados y controles sobreestima considerablemente el ATE real.

El doctor ingenuo asigna el tratamiento de forma aleatoria, sin importar quién se beneficia más. Dado que la asignación es independiente de los resultados potenciales, la diferencia de medias observada es un estimador insesgado del ATE, aunque con varianza mayor.

El punto central es que el mecanismo de asignación no modifica los resultados potenciales ni el ATE verdadero; solo determina si lo que observamos en los datos permite estimarlo de forma válida. Un mecanismo que depende de los resultados potenciales introduce sesgo de selección.